In [ ]:
!pip install langchain langchain-community langgraph transformers accelerate bitsandbytes peft duckduckgo-search trafilatura


In [ ]:
!pip install yfinance GoogleNews langchain-huggingface langchain_community chromadb sentence-transformers pandas tqdm

In [ ]:
%%capture

!pip install -U langchain langchain-community langchain-core transformers accelerate bitsandbytes trafilatura duckduckgo-search
!pip install --upgrade jupyter_client ipykernel

import warnings
import logging
import os
import sys

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

sys.stderr = open(os.devnull, "w")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_community.llms import HuggingFacePipeline

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16
)



In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.3,
)

llm = HuggingFacePipeline(pipeline=pipe)
print("Model incarcat cu succes!")

In [ ]:
!pip install langchain-huggingface

In [ ]:
import os
import pandas as pd
import datetime
import time
import yfinance as yf
from tqdm import tqdm
from GoogleNews import GoogleNews
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma(embedding_function=embeddings, persist_directory="./trading_memory_db")

print(" Sistemul de memorie este pregătit!")

In [ ]:
def ingest_kaggle_csv(csv_path, ticker_filter=None, limit=5000):
    print(f" Încărcare date istorice din CSV: {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
        df.columns = [c.lower() for c in df.columns]

        col_text = next((c for c in ['headline', 'title', 'news'] if c in df.columns), None)
        col_date = next((c for c in ['date', 'timestamp'] if c in df.columns), None)
        col_stock = next((c for c in ['stock', 'ticker'] if c in df.columns), None)

        if ticker_filter and col_stock:
            df = df[df[col_stock] == ticker_filter]

        documents = []
        print(f"    Procesare {min(len(df), limit)} știri...")

        for _, row in tqdm(df.head(limit).iterrows(), total=min(len(df), limit)):
            try:
                date_str = str(row[col_date])[:10]
                ts = int(pd.to_datetime(date_str).timestamp())

                doc = Document(
                    page_content=str(row[col_text]),
                    metadata={
                        "date": date_str,
                        "timestamp": ts,
                        "source": "Kaggle_Archive",
                        "ticker": ticker_filter if ticker_filter else "General"
                    }
                )
                documents.append(doc)
            except: continue

        if documents:
            for i in range(0, len(documents), 500):
                vector_db.add_documents(documents[i:i+500])
        print(" Datele istorice (Kaggle) au fost salvate!")

    except Exception as e:
        print(f" Eroare CSV: {e}")

TRUSTED_SOURCES = ["reuters", "bloomberg", "cnbc", "finance.yahoo", "wsj", "marketwatch"]

def ingest_trusted_recent_news(ticker, start_year=2021):
    print(f" Descărcare date recente (Trusted Sources) pentru {ticker}...")
    googlenews = GoogleNews(lang='en', region='US')
    start_date = datetime.date(start_year, 1, 1)
    end_date = datetime.date.today()

    current_date = start_date
    documents = []

    pbar = tqdm(total=(end_date - start_date).days // 7)

    while current_date < end_date:
        next_week = current_date + datetime.timedelta(days=7)
        try:
            googlenews.set_time_range(current_date.strftime("%m/%d/%Y"), next_week.strftime("%m/%d/%Y"))
            googlenews.search(f"{ticker} stock")
            results = googlenews.result()
            googlenews.clear()

            for item in results:
                if any(src in item.get('link', '').lower() for src in TRUSTED_SOURCES):
                    ts = int(current_date.strftime("%s"))
                    doc = Document(
                        page_content=item['title'],
                        metadata={
                            "date": current_date.strftime("%Y-%m-%d"),
                            "timestamp": ts,
                            "source": "GoogleNews_Recent",
                            "ticker": ticker
                        }
                    )
                    documents.append(doc)

            if len(documents) > 20:
                vector_db.add_documents(documents)
                documents = []

        except: pass
        current_date = next_week
        pbar.update(1)
        time.sleep(1.5)

    if documents: vector_db.add_documents(documents)
    pbar.close()
    print("Datele recente au fost actualizate!")

In [ ]:

news_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.15
)

news_llm = HuggingFacePipeline(pipeline=news_pipe)

news_agent_prompt = PromptTemplate.from_template(
    "[INST] You are an Objective Financial Data Miner.\n"
    "Your goal is to read the article below and extract specific data points.\n"
    "DO NOT interpret or summarize. Just extract facts.\n\n"

    "Article Content:\n"
    "{text}\n\n"

    "Instructions for the Report:\n"
    "1. First, write the TITLE of the article.\n"
    "2. Create a section called 'SECTION 1: HARD NUMBERS'. List every specific financial number you find (Revenue, EPS, Growth %, Guidance numbers). Write them exactly as they appear.\n"
    "3. Create a section called 'SECTION 2: KEY DEVELOPMENTS'. List factual events (launches, restrictions, delays).\n"
    "4. Create a section called 'SECTION 3: DIRECT QUOTES'. Extract 2-3 important sentences spoken by executives.\n"
    "5. Create a section called 'SECTION 4: CONTEXT'. List the company tickers and any external factors mentions.\n\n"

    "IMPORTANT: Do not use placeholders like '[Insert here]'. Write the ACTUAL data directly.\n"
    "[/INST]\n\n"

    "### REPORT START\n"
    "TITLE:"
)


news_chain = RunnableSequence(news_agent_prompt, news_llm, StrOutputParser())

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

tech_prompt = PromptTemplate.from_template(
    "[INST] You are a Technical Analyst. Analyze the price data below.\n"
    "DATA: {price_data}\n"
    "TASK: Identify trend (Bull/Bear), Volatility, and give a signal (BUY/SELL/WAIT).\n"
    "[/INST]"
)
technical_chain = tech_prompt | llm | StrOutputParser()

correlation_prompt = PromptTemplate.from_template(
    "[INST] Analyze the relationship between News and Price.\n"
    "NEWS: {news}\n"
    "ACTUAL PRICE MOVE: {move}%\n"
    "Did the news cause this move? Rate correlation 0-10.\n"
    "[/INST]"
)
correlation_chain = correlation_prompt | llm | StrOutputParser()

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser


bull_round1_prompt = PromptTemplate.from_template(
    "You are a financial analyst with a BULLISH perspective.\n"
    "Analyze the following text and extract ONLY positive arguments.\n"
    "-----BEGIN TEXT-----\n{text}\n-----END TEXT-----\n\n"
    "List the positive arguments below:\n"
    "### OUTPUT"
)

bear_round1_prompt = PromptTemplate.from_template(
    "You are a financial analyst with a BEARISH perspective.\n"
    "Analyze the following text and extract ONLY negative arguments.\n"
    "-----BEGIN TEXT-----\n{text}\n-----END TEXT-----\n\n"
    "List the negative arguments below:\n"
    "### OUTPUT"
)

bull_reply_prompt = PromptTemplate.from_template(
    "You are a BULLISH trader debating a Bearish opponent.\n"
    "Context: {text}\n"
    "Opponent's Arguments: {bear}\n\n"
    "Provide counter-arguments to defend your position.\n"
    "### OUTPUT"
)

bear_reply_prompt = PromptTemplate.from_template(
    "You are a BEARISH trader debating a Bullish opponent.\n"
    "Context: {text}\n"
    "Opponent's Arguments: {bull}\n\n"
    "Provide counter-arguments to defend your position.\n"
    "### OUTPUT"
)

judge_prompt = PromptTemplate.from_template(
    "You are a Chief Investment Officer.\n"
    "Review the debate arguments below:\n"
    "Bullish: {bull1} | Reply: {bull2}\n"
    "Bearish: {bear1} | Reply: {bear2}\n\n"
    "Decide the winner and explain why.\n"
    "### OUTPUT"
)

decision_prompt = PromptTemplate.from_template(
    "Based on the debate winner: {winner} and text: {text}\n"
    "Make a final decision (BUY, SELL, or HOLD) and explain strictly why.\n"
    "### OUTPUT"
)


In [ ]:
bull_round1 = RunnableSequence(bull_round1_prompt, llm, StrOutputParser())
bear_round1 = RunnableSequence(bear_round1_prompt, llm, StrOutputParser())
bull_reply = RunnableSequence(bull_reply_prompt, llm, StrOutputParser())
bear_reply = RunnableSequence(bear_reply_prompt, llm, StrOutputParser())
judge_agent = RunnableSequence(judge_prompt, llm, StrOutputParser())
decision_agent = RunnableSequence(decision_prompt, llm, StrOutputParser())

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

class DebateState(TypedDict):
    text: str
    bull1: Optional[str]
    bear1: Optional[str]
    bull2: Optional[str]
    bear2: Optional[str]
    judge: Optional[str]
    decision: Optional[str]

def step_bull1(state):
    print("\n [1/6] BULLISH Agent (Round 1) is thinking...")
    raw = bull_round1.invoke({"text": state["text"]})

    print(f"--- RAW OUTPUT BULL 1 ---\n{raw}\n-------------------------")

    return {"bull1": extract_block(raw)}

def step_bear1(state):
    print("\n [2/6] BEARISH Agent (Round 1) is thinking...")
    raw = bear_round1.invoke({"text": state["text"]})

    print(f"--- RAW OUTPUT BEAR 1 ---\n{raw}\n-------------------------")

    return {"bear1": extract_block(raw)}

def step_bull2(state):
    print("\n [3/6] BULLISH Agent (Reply) is countering...")
    raw = bull_reply.invoke({
        "text": state["text"],
        "bear": state["bear1"]
    })

    print(f"--- RAW OUTPUT BULL 2 ---\n{raw}\n-------------------------")

    return {"bull2": extract_block(raw)}

def step_bear2(state):
    print("\n [4/6] BEARISH Agent (Reply) is countering...")
    raw = bear_reply.invoke({
        "text": state["text"],
        "bull": state["bull1"]
    })

    print(f"--- RAW OUTPUT BEAR 2 ---\n{raw}\n-------------------------")

    return {"bear2": extract_block(raw)}

def step_judge(state):
    print("\n [5/6] JUDGE is deliberating...")
    raw = judge_agent.invoke({
        "bull1": state["bull1"],
        "bear1": state["bear1"],
        "bull2": state["bull2"],
        "bear2": state["bear2"]
    })

    print(f"--- RAW OUTPUT JUDGE ---\n{raw}\n------------------------")

    return {"judge": extract_block(raw)}

def step_decision(state):
    print("\n [6/6] FINAL DECISION is being made...")

    winner_text = state["judge"].lower()
    if "bullish" in winner_text and "bearish" not in winner_text:
        winner = "BULLISH"
    elif "bearish" in winner_text and "bullish" not in winner_text:
        winner = "BEARISH"
    else:
        winner = "UNCERTAIN"

    raw = decision_agent.invoke({
        "winner": winner,
        "text": state["text"]
    })

    print(f"--- RAW OUTPUT DECISION ---\n{raw}\n---------------------------")

    return {"decision": extract_block(raw)}

workflow = StateGraph(DebateState)
workflow.add_node("bull1", step_bull1)
workflow.add_node("bear1", step_bear1)
workflow.add_node("bull2", step_bull2)
workflow.add_node("bear2", step_bear2)
workflow.add_node("judge", step_judge)
workflow.add_node("decision", step_decision)

workflow.set_entry_point("bull1")

workflow.add_edge("bull1", "bear1")
workflow.add_edge("bear1", "bull2")
workflow.add_edge("bull2", "bear2")
workflow.add_edge("bear2", "judge")
workflow.add_edge("judge", "decision")
workflow.add_edge("decision", END)

graph = workflow.compile()
